# Phase 1: Single Image Analysis - Verification Notebook

This notebook verifies the Phase 1 implementation of the pickleball pose analyzer step by step.

## Overview

Phase 1 focuses on:
1. ✅ Loading and initializing the SAM 3D Body model
2. ✅ Processing individual pickleball pose images
3. ✅ Extracting 3D keypoints and pose data
4. ✅ Scoring each position (preparation, contact, finish)

## Key APIs

- **`load_sam3d_model()`** - Initialize SAM 3D Body estimator
- **`process_single_image()`** - Process one image and extract pose data
- **`process_three_positions()`** - Process three images at once
- **`extract_keypoints()`** - Extract specific keypoints by name
- **`score_preparation_position()`** - Score preparation pose
- **`score_contact_position()`** - Score contact pose
- **`score_finish_position()`** - Score finish pose

## Step 1: Setup and Imports

Import the pickleball_pose_analyzer module and verify it's working.

In [3]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent.parent
sys.path.insert(0, str(project_root / 'libraries' / 'src'))

# Import pickleball_pose_analyzer
from pickleball_pose_analyzer import (
    # Model loading
    load_sam3d_model,
    # Image processing
    process_single_image,
    process_three_positions,
    # Keypoint extraction
    extract_keypoints,
    calculate_body_center,
    # Scoring
    score_preparation_position,
    score_contact_position,
    score_finish_position,
    calculate_3d_angle,
    # Data structures
    PICKLEBALL_KEYPOINTS,
    KEYPOINT_NAMES_TO_INDICES,
    PoseData,
    PositionScore,
)

print("✅ Successfully imported pickleball_pose_analyzer")
print(f"Available keypoints: {list(PICKLEBALL_KEYPOINTS.keys())}")

✅ Successfully imported pickleball_pose_analyzer
Available keypoints: ['left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist', 'neck', 'left_hip', 'right_hip', 'left_knee', 'right_knee', 'left_ankle', 'right_ankle', 'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear']


## Step 2: Load SAM 3D Body Model

**Key API: `load_sam3d_model()`**

This function supports loading from:
- **HuggingFace** (default): Downloads and caches the model in `~/.cache/huggingface/hub/`
- **Local checkpoint**: Load from a local checkpoint file or directory

**Local Checkpoint Structure:**
```
checkpoints/sam-3d-body-dinov3/
  ├── model.ckpt              # Main model checkpoint
  ├── assets/
  │   └── mhr_model.pt       # MHR (Momentum Human Rig) model
  └── model_config.yaml       # Model configuration (auto-detected)
```

**Parameters:**
- `checkpoint_path`: Local path to checkpoint file (e.g., `"model.ckpt"`) or directory containing `"model.ckpt"`. If provided, loads from local checkpoint instead of HuggingFace.
- `hf_repo_id`: HuggingFace repository ID (default: "facebook/sam-3d-body-dinov3", used if `checkpoint_path` is None)
- `device`: "auto", "cuda", or "cpu" (default: "auto")
- `use_detector`: Whether to use human detector (default: True)
- `use_fov_estimator`: Whether to use FOV estimator (default: True)

In [4]:
# Option 1: Load from HuggingFace (default)
"""
print("=" * 60)
print("Option 1: Loading from HuggingFace")
print("=" * 60)
print("This may take a few minutes on first run (downloading from HuggingFace)")

estimator, config = load_sam3d_model(
    hf_repo_id="facebook/sam-3d-body-dinov3",
    device="auto",  # Auto-detect CUDA if available
    use_detector=True,
    use_fov_estimator=True,
)

print(f"✅ Model loaded successfully!")
print(f"   Source: HuggingFace ({config.get('hf_repo_id', 'unknown')})")
print(f"   Device: {config.get('device', 'unknown')}")
print(f"   Has detector: {config.get('has_detector', False)}")
print(f"   Has FOV estimator: {config.get('has_fov_estimator', False)}")
print()
"""
# Option 2: Load from local checkpoint (if available)
print("=" * 60)
print("Option 2: Loading from Local Checkpoint")
print("=" * 60)

# Example: Load from a checkpoint directory or file
# Option A: Specify checkpoint directory (function will look for model.ckpt inside)
checkpoint_path = project_root / 'models' / 'sam-3d-body' / 'checkpoints' / 'sam-3d-body-dinov3'

# Option B: Specify checkpoint file directly (uncomment to use):
# checkpoint_path = project_root / 'models' / 'sam-3d-body' / 'checkpoints' / 'sam-3d-body-dinov3' / 'model.ckpt'

# Option C: Use absolute path (uncomment and adjust if needed):
# checkpoint_path = Path("/Users/chang/Documents/dev/git/ml/coachKata/models/sam-3d-body/checkpoints/sam-3d-body-dinov3")

# Check if checkpoint exists (as Path object)
if checkpoint_path.exists():
    print(f"Found checkpoint at: {checkpoint_path}")
    estimator, config = load_sam3d_model(
        checkpoint_path=str(checkpoint_path),  # Convert to string for the function
        device="auto",
        use_detector=True,
        use_fov_estimator=True,
    )
    
    print(f"✅ Model loaded successfully!")
    print(f"   Source: Local checkpoint ({config.get('checkpoint_path', 'unknown')})")
    print(f"   Device: {config.get('device', 'unknown')}")
    print(f"   Has detector: {config.get('has_detector', False)}")
    print(f"   Has FOV estimator: {config.get('has_fov_estimator', False)}")
else:
    print(f"⚠️  Local checkpoint not found at: {checkpoint_path}")
    print("   Please download checkpoints or use Option 1 (HuggingFace) instead.")
    print()
    print("   To download checkpoints from HuggingFace:")
    print("   hf download facebook/sam-3d-body-dinov3 --local-dir models/sam-3d-body/checkpoints/sam-3d-body-dinov3")


Option 2: Loading from Local Checkpoint
Found checkpoint at: /Users/chang/Documents/dev/git/ml/coachKata/models/sam-3d-body/checkpoints/sam-3d-body-dinov3
CUDA not available, using CPU
Using device: cpu
Loading SAM 3D Body model from local checkpoint: /Users/chang/Documents/dev/git/ml/coachKata/models/sam-3d-body/checkpoints/sam-3d-body-dinov3
Loading SAM 3D Body model...


Using cache found in /Users/chang/.cache/torch/hub/facebookresearch_dinov3_main
Ignored kwargs: {'drop_path': 0.1}
The model and loaded state dict do not match exactly

missing keys in source state_dict: backbone.encoder.mask_token, head_pose.hand_pose_comps_ori, head_pose.mhr.face_expressions_model.shape_vectors, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_indices, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.0.sparse_weight, head_pose.mhr.pose_correctives_model.pose_dirs_predictor.2.weight, head_pose.mhr.character_torch.skeleton.joint_translation_offsets, head_pose.mhr.character_torch.skeleton.joint_prerotations, head_pose.mhr.character_torch.skeleton.pmi, head_pose.mhr.character_torch.skeleton.joint_parents, head_pose.mhr.character_torch.mesh.rest_vertices, head_pose.mhr.character_torch.mesh.faces, head_pose.mhr.character_torch.mesh.texcoords, head_pose.mhr.character_torch.mesh.texcoord_faces, head_pose.mhr.character_torch.parameter_transform.parame

Loading human detector: vitdet...
########### Using human detector: ViTDet...


/Users/chang/Documents/dev/git/ml/coachKata/.venv/lib/python3.11/site-packages/detectron2/config/lazy.py:167: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return old_import(name, globals, locals, fromlist=fromlist, level=level)
/Users/chang/Documents/dev/git/ml/coachKata/.venv/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading FOV estimator: moge2...
########### Using fov estimator: MoGe2...


model.pt:   0%|          | 0.00/1.32G [00:00<?, ?B/s]

Mask-condition inference is not supported...
Model loaded successfully!
✅ Model loaded successfully!
   Source: Local checkpoint (/Users/chang/Documents/dev/git/ml/coachKata/models/sam-3d-body/checkpoints/sam-3d-body-dinov3/model.ckpt)
   Device: cpu
   Has detector: True
   Has FOV estimator: True


## Step 3: Process a Single Image

**Key API: `process_single_image()`**

This function:
- Loads an image from file path
- Processes it through SAM 3D Body estimator
- Extracts 3D keypoints, vertices, pose parameters, and metadata
- Returns structured pose data dictionary

**Returns:**
- `pred_keypoints_3d`: (70, 3) array of 3D keypoints
- `pred_keypoints_2d`: (70, 2) array of 2D keypoints
- `pred_vertices`: (18439, 3) mesh vertices
- `body_pose_params`: Body pose parameters
- `hand_pose_params`: Hand pose parameters
- `shape_params`: Shape parameters
- `scale_params`: Scale parameters
- `pred_global_rots`: (127, 3, 3) global rotation matrices
- `bbox`: Bounding box
- `focal_length`: Estimated focal length
- `metadata`: Image metadata

In [5]:
# Set up image paths (adjust these to your actual image paths)
data_dir = project_root / 'data' / 'pickelball'

# Example: Use one of the available images
# You can change this to any pickleball pose image
image_path = data_dir / 'bh-1.png'

if not image_path.exists():
    print(f"⚠️  Image not found: {image_path}")
    print("Please update the image_path variable with a valid image path")
else:
    print(f"Processing image: {image_path}")
    
    # Process single image
    pose_data = process_single_image(
        estimator=estimator,
        image_path=str(image_path),
        position_name="test",
        bbox_thr=0.5,
        use_mask=False,
    )
    
    print(f"✅ Image processed successfully!")
    print(f"   Position: {pose_data['position_name']}")
    print(f"   3D Keypoints shape: {pose_data['pred_keypoints_3d'].shape}")
    print(f"   2D Keypoints shape: {pose_data['pred_keypoints_2d'].shape}")
    print(f"   Vertices shape: {pose_data['pred_vertices'].shape}")
    print(f"   Focal length: {pose_data['focal_length']:.2f}")
    print(f"   Bbox: {pose_data['bbox']}")

Processing image: /Users/chang/Documents/dev/git/ml/coachKata/data/pickelball/bh-1.png
Running object detector...


/Users/chang/Documents/dev/git/ml/coachKata/.venv/lib/python3.11/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4324.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Found boxes: [[3.0108616e-01 3.8060482e+01 2.3057689e+02 3.8857626e+02]
 [1.5732153e+02 5.0519114e+00 2.1322534e+02 1.7242809e+02]]
Running FOV estimator ...


/Users/chang/Documents/dev/git/ml/coachKata/.venv/lib/python3.11/site-packages/torch/amp/autocast_mode.py:283: UserWarning: In CPU autocast, but the target dtype is not supported. Disabling autocast.
CPU Autocast only supports dtype of torch.bfloat16, torch.float16 currently.
  warnings.warn(error_message)


✅ Image processed successfully!
   Position: test
   3D Keypoints shape: (70, 3)
   2D Keypoints shape: (70, 2)
   Vertices shape: (18439, 3)
   Focal length: 988.19
   Bbox: [3.0108616e-01 3.8060482e+01 2.3057689e+02 3.8857626e+02]


## Step 4: Extract Keypoints

**Key API: `extract_keypoints()`**

This function extracts specific keypoints by name from the pose data.

**Available keypoints:**
- Upper body: `left_shoulder`, `right_shoulder`, `left_elbow`, `right_elbow`, `left_wrist`, `right_wrist`, `neck`
- Lower body: `left_hip`, `right_hip`, `left_knee`, `right_knee`, `left_ankle`, `right_ankle`
- Head: `nose`, `left_eye`, `right_eye`, `left_ear`, `right_ear`

**Key API: `calculate_body_center()`**

Calculates the center of the body using hip keypoints.

In [6]:
# Extract keypoints by name
keypoints = extract_keypoints(pose_data)

print("✅ Extracted keypoints:")
print(f"   Available keypoints: {len(keypoints)}")
print()

# Show some example keypoints
example_keypoints = ['left_shoulder', 'right_shoulder', 'left_hip', 'right_hip', 'left_wrist', 'right_wrist']
for name in example_keypoints:
    if name in keypoints and keypoints[name] is not None:
        coords = keypoints[name]
        print(f"   {name:20s}: [{coords[0]:7.3f}, {coords[1]:7.3f}, {coords[2]:7.3f}]")
    else:
        print(f"   {name:20s}: Not available")

# Calculate body center
body_center = calculate_body_center(pose_data)
print()
print(f"✅ Body center: [{body_center[0]:7.3f}, {body_center[1]:7.3f}, {body_center[2]:7.3f}]")

✅ Extracted keypoints:
   Available keypoints: 18

   left_shoulder       : [ -0.071,  -1.323,   0.347]
   right_shoulder      : [  0.171,  -1.346,   0.181]
   left_hip            : [ -0.080,  -0.907,   0.017]
   right_hip           : [  0.078,  -0.899,  -0.011]
   left_wrist          : [  0.384,  -1.457,   0.559]
   right_wrist         : [  0.486,  -1.385,   0.462]

✅ Body center: [ -0.001,  -0.903,   0.003]


## Step 5: Score Preparation Position

**Key API: `score_preparation_position()`**

Scores the preparation position based on:
- **Shoulder angle**: Angle between shoulders and hips (should be open)
- **Weight distribution**: Balance between left and right hips
- **Knee bend**: Angle of knee joints (should be slightly bent)

**Returns:**
- Individual metric scores (0-100)
- Weighted overall preparation score (0-100)

In [7]:
# Score preparation position
prep_score = score_preparation_position(pose_data)

print("✅ Preparation Position Score:")
print(f"   Overall Score: {prep_score['preparation_score']:.2f}/100")
print()
print("   Individual Metrics:")
print(f"   - Shoulder Angle Score: {prep_score['scores']['shoulder_angle']:.2f}/100")
print(f"     (Right: {prep_score['shoulder_angle_right']:.1f}°, Left: {prep_score['shoulder_angle_left']:.1f}°)")
print(f"   - Weight Distribution Score: {prep_score['scores']['weight_distribution']:.2f}/100")
print(f"     (Hip height difference: {prep_score['hip_height_diff']*100:.2f} cm)")
print(f"   - Knee Bend Score: {prep_score['scores']['knee_bend']:.2f}/100")
print(f"     (Left knee: {prep_score['knee_angles']['left']:.1f}°, Right knee: {prep_score['knee_angles']['right']:.1f}°)")

✅ Preparation Position Score:
   Overall Score: 90.00/100

   Individual Metrics:
   - Shoulder Angle Score: 100.00/100
     (Right: 91.9°, Left: 115.2°)
   - Weight Distribution Score: 100.00/100
     (Hip height difference: 2.87 cm)
   - Knee Bend Score: 100.00/100
     (Left knee: 145.2°, Right knee: 153.9°)


## Step 6: Score Contact Position

**Key API: `score_contact_position()`**

Scores the contact position (point of contact between paddle and ball) based on:
- **Paddle position**: Wrist position relative to body (should be in front of body)
- **Contact height**: Height of paddle at contact point
- **Body alignment**: Shoulder and hip alignment in frontal plane
- **Torso angle**: Forward lean angle (optional metric)

**Returns:**
- Individual metric scores (0-100)
- Weighted overall contact score (0-100)

In [11]:
contact_score

{'paddle_position': array([ 0.48613325, -0.48241222,  0.45883417], dtype=float32),
 'paddle_height': 0.4617008566856384,
 'torso_angle': 31.017148913645222,
 'body_alignment': 0.0,
 'scores': {'paddle_position': 100.0,
  'contact_height': 17.712607085704782,
  'body_alignment': 0.0,
  'torso_angle': 0.0},
 'contact_score': 39.428151771426194}

In [9]:
contact_score['scores']

{'paddle_position': 100.0,
 'contact_height': 17.712607085704782,
 'body_alignment': 0.0,
 'torso_angle': 0.0}

In [15]:
# Score contact position
contact_score = score_contact_position(pose_data)

print("✅ Contact Position Score:")
print(f"   Overall Score: {contact_score['contact_score']:.2f}/100")
print()
print("   Individual Metrics:")
print(f"   - Paddle Position Score: {contact_score['scores']['paddle_position']:.2f}/100")
print(f"   - Body Alignment Score: {contact_score['scores']['body_alignment']:.2f}/100")
print(f"   - Contact Height Score: {contact_score['scores']['contact_height']:.2f}/100")
print(f"     (Paddle height: {contact_score['paddle_height']:.3f}m)")
if contact_score.get('torso_angle') is not None:
    print(f"   - Torso Angle Score: {contact_score['scores'].get('torso_angle', 0.0):.2f}/100")
    print(f"     (Torso angle: {contact_score['torso_angle']:.1f}°)")

✅ Contact Position Score:
   Overall Score: 39.43/100

   Individual Metrics:
   - Paddle Position Score: 100.00/100
   - Body Alignment Score: 0.00/100
   - Contact Height Score: 17.71/100
     (Paddle height: 0.462m)
   - Torso Angle Score: 0.00/100
     (Torso angle: 31.0°)


## Step 7: Score Finish Position

**Key API: `score_finish_position()`**

Scores the finish position (end of swing) based on:
- **Finish position**: Wrist position and height at finish
- **Follow-through angle**: Shoulder-elbow-wrist angle (full arm extension)
- **Body rotation**: Hip and shoulder rotation alignment

**Returns:**
- Individual metric scores (0-100)
- Weighted overall finish score (0-100)

In [20]:
# Score finish position
finish_score = score_finish_position(pose_data)

print("✅ Finish Position Score:")
print(f"   Overall Score: {finish_score['finish_score']:.2f}/100")
print()
print("   Individual Metrics:")
print(f"   - Finish Position Score: {finish_score['scores']['finish_position']:.2f}/100")
print(f"     (Finish position: {finish_score['finish_position']})")
print(f"   - Follow-through Angle Score: {finish_score['scores']['follow_through']:.2f}/100")
print(f"     (Follow-through angle: {finish_score['follow_through_angle']:.1f}°)")
print(f"   - Body Rotation Score: {finish_score['scores']['body_rotation']:.2f}/100")
print(f"     (Body rotation metric: {finish_score['body_rotation']:.3f})")

✅ Finish Position Score:
   Overall Score: 8.72/100

   Individual Metrics:
   - Finish Position Score: 6.17/100
     (Finish position: [ 0.48554593 -1.3853569   0.46170086])
   - Follow-through Angle Score: 0.00/100
     (Follow-through angle: 108.2°)
   - Body Rotation Score: 25.00/100
     (Body rotation metric: 0.250)


## Step 8: Process Three Positions Together

**Key API: `process_three_positions()`**

This function processes three images (preparation, contact, finish) in one call and returns structured data for all three positions.

**Parameters:**
- `estimator`: SAM3DBodyEstimator instance
- `preparation_path`: Path to preparation image
- `contact_path`: Path to contact image
- `finish_path`: Path to finish image

**Returns:**
- Dictionary with keys: `'preparation'`, `'contact'`, `'finish'`
- Each contains the same structure as `process_single_image()` output

In [21]:
# Example: Process three positions
# Update these paths to your actual pickleball pose images
preparation_path = data_dir / 'drop-shovel-1.png'  # Update as needed
contact_path = data_dir / 'drop-shovel-5.png'      # Update as needed
finish_path = data_dir / 'drop-shovel-8.png'        # Update as needed

# Check if images exist
images_exist = all(p.exists() for p in [preparation_path, contact_path, finish_path])

if images_exist:
    print("Processing three positions...")
    print(f"   Preparation: {preparation_path.name}")
    print(f"   Contact: {contact_path.name}")
    print(f"   Finish: {finish_path.name}")
    print()
    
    # Process all three positions
    results = process_three_positions(
        estimator=estimator,
        preparation_path=str(preparation_path),
        contact_path=str(contact_path),
        finish_path=str(finish_path),
    )
    
    print("✅ All three positions processed!")
    print()
    
    # Score each position
    prep_score = score_preparation_position(results['preparation'])
    contact_score = score_contact_position(results['contact'])
    finish_score = score_finish_position(results['finish'])
    
    # Display summary
    print("=" * 60)
    print("SCORING SUMMARY")
    print("=" * 60)
    print(f"Preparation Score: {prep_score['preparation_score']:.2f}/100")
    print(f"Contact Score:     {contact_score['contact_score']:.2f}/100")
    print(f"Finish Score:       {finish_score['finish_score']:.2f}/100")
    print()
    
    # Calculate cumulative score (average of all three)
    cumulative_score = (
        prep_score['preparation_score'] +
        contact_score['contact_score'] +
        finish_score['finish_score']
    ) / 3.0
    print(f"Cumulative Score:   {cumulative_score:.2f}/100")
    print("=" * 60)
else:
    print("⚠️  Some images not found. Please update the image paths above.")
    print("   You need three images: preparation, contact, and finish positions")

Processing three positions...
   Preparation: drop-shovel-1.png
   Contact: drop-shovel-5.png
   Finish: drop-shovel-8.png

Processing preparation position...
Running object detector...
Found boxes: [[ 20.44346    9.529263 132.10425  238.80573 ]]
Running FOV estimator ...


/Users/chang/Documents/dev/git/ml/coachKata/.venv/lib/python3.11/site-packages/torch/amp/autocast_mode.py:283: UserWarning: In CPU autocast, but the target dtype is not supported. Disabling autocast.
CPU Autocast only supports dtype of torch.bfloat16, torch.float16 currently.
  warnings.warn(error_message)


Processing contact position...
Running object detector...
Found boxes: [[ 77.81872   23.316877 207.74971  224.38869 ]]
Running FOV estimator ...


/Users/chang/Documents/dev/git/ml/coachKata/.venv/lib/python3.11/site-packages/torch/amp/autocast_mode.py:283: UserWarning: In CPU autocast, but the target dtype is not supported. Disabling autocast.
CPU Autocast only supports dtype of torch.bfloat16, torch.float16 currently.
  warnings.warn(error_message)


Processing finish position...
Running object detector...
Found boxes: [[ 65.10234   17.458778 190.82262  244.90942 ]]
Running FOV estimator ...


/Users/chang/Documents/dev/git/ml/coachKata/.venv/lib/python3.11/site-packages/torch/amp/autocast_mode.py:283: UserWarning: In CPU autocast, but the target dtype is not supported. Disabling autocast.
CPU Autocast only supports dtype of torch.bfloat16, torch.float16 currently.
  warnings.warn(error_message)


✅ All three positions processed!

SCORING SUMMARY
Preparation Score: 30.67/100
Contact Score:     8.75/100
Finish Score:       18.75/100

Cumulative Score:   19.39/100


## Step 9: Calculate 3D Angles

**Key API: `calculate_3d_angle()`**

Utility function to calculate the angle between three 3D points (useful for joint angles).

**Parameters:**
- `point1`, `point2`, `point3`: Three 3D points (numpy arrays)
- `point2` is the vertex of the angle

**Returns:**
- Angle in degrees (0-180°)

In [22]:
# Example: Calculate elbow angle
if 'pose_data' in locals():
    keypoints = extract_keypoints(pose_data)
    
    # Calculate left elbow angle (shoulder-elbow-wrist)
    if all(k in keypoints and keypoints[k] is not None 
           for k in ['left_shoulder', 'left_elbow', 'left_wrist']):
        elbow_angle = calculate_3d_angle(
            keypoints['left_shoulder'],
            keypoints['left_elbow'],
            keypoints['left_wrist']
        )
        print(f"✅ Left elbow angle: {elbow_angle:.1f}°")
    
    # Calculate right elbow angle
    if all(k in keypoints and keypoints[k] is not None 
           for k in ['right_shoulder', 'right_elbow', 'right_wrist']):
        elbow_angle = calculate_3d_angle(
            keypoints['right_shoulder'],
            keypoints['right_elbow'],
            keypoints['right_wrist']
        )
        print(f"✅ Right elbow angle: {elbow_angle:.1f}°")
    
    # Calculate knee angle (hip-knee-ankle)
    if all(k in keypoints and keypoints[k] is not None 
           for k in ['left_hip', 'left_knee', 'left_ankle']):
        knee_angle = calculate_3d_angle(
            keypoints['left_hip'],
            keypoints['left_knee'],
            keypoints['left_ankle']
        )
        print(f"✅ Left knee angle: {knee_angle:.1f}°")

✅ Left elbow angle: 153.7°
✅ Right elbow angle: 108.2°
✅ Left knee angle: 145.2°


## Step 10: Data Structures

**Key Data Classes:**

- **`PoseData`**: Structured container for pose data
  - `keypoints_3d`: 3D keypoints array
  - `keypoints_2d`: 2D keypoints array
  - `vertices`: Mesh vertices
  - `pose_params`: Pose parameters
  - `metadata`: Image metadata

- **`PositionScore`**: Container for position scores
  - `overall_score`: Overall score (0-100)
  - `metric_scores`: Dictionary of individual metric scores
  - `raw_metrics`: Raw metric values

- **`PICKLEBALL_KEYPOINTS`**: Dictionary mapping keypoint names to indices
- **`KEYPOINT_NAMES_TO_INDICES`**: Reverse mapping for easy lookup

In [23]:
# Display available keypoints
print("Available Pickleball Keypoints:")
print("=" * 40)
for name, idx in sorted(PICKLEBALL_KEYPOINTS.items(), key=lambda x: x[1]):
    print(f"  {name:20s} -> index {idx:2d}")

print()
print(f"Total keypoints: {len(PICKLEBALL_KEYPOINTS)}")

Available Pickleball Keypoints:
  nose                 -> index  0
  left_eye             -> index  1
  right_eye            -> index  2
  left_ear             -> index  3
  right_ear            -> index  4
  left_shoulder        -> index  5
  right_shoulder       -> index  6
  left_elbow           -> index  7
  right_elbow          -> index  8
  left_hip             -> index  9
  right_hip            -> index 10
  left_knee            -> index 11
  right_knee           -> index 12
  left_ankle           -> index 13
  right_ankle          -> index 14
  right_wrist          -> index 41
  left_wrist           -> index 62
  neck                 -> index 69

Total keypoints: 18


## Summary

This notebook verified the Phase 1 implementation:

✅ **Model Loading**: Successfully load SAM 3D Body from HuggingFace or local checkpoints  
✅ **Image Processing**: Process single images and extract pose data  
✅ **Keypoint Extraction**: Extract specific keypoints by name  
✅ **Position Scoring**: Score preparation, contact, and finish positions  
✅ **Batch Processing**: Process three positions together  
✅ **Utility Functions**: Calculate 3D angles and body center  

## Next Steps (Phase 2)

Phase 2 will add:
- Pose comparison between reference and student poses
- Detailed feedback generation
- Visualization of pose differences
- Advanced scoring metrics

## API Reference Quick Guide

### Model Loading

**From HuggingFace (default):**
```python
estimator, config = load_sam3d_model(
    hf_repo_id="facebook/sam-3d-body-dinov3",
    device="auto",
    use_detector=True,
    use_fov_estimator=True,
)
```

**From Local Checkpoint:**
```python
# Option 1: Specify checkpoint directory (auto-finds model.ckpt)
estimator, config = load_sam3d_model(
    checkpoint_path="path/to/checkpoints/sam-3d-body-dinov3",
    device="auto",
    use_detector=True,
    use_fov_estimator=True,
)

# Option 2: Specify checkpoint file directly
estimator, config = load_sam3d_model(
    checkpoint_path="path/to/checkpoints/sam-3d-body-dinov3/model.ckpt",
    device="auto",
    use_detector=True,
    use_fov_estimator=True,
)
```

**Local Checkpoint Structure:**
The function automatically looks for:
- `model.ckpt` - Main model checkpoint (required)
- `assets/mhr_model.pt` - MHR model (optional, will warn if not found)
- `model_config.yaml` - Model config (auto-detected by underlying function)

### Image Processing
```python
# Single image
pose_data = process_single_image(estimator, image_path, position_name="preparation")

# Three positions
results = process_three_positions(estimator, prep_path, contact_path, finish_path)
```

### Keypoint Extraction
```python
keypoints = extract_keypoints(pose_data)
body_center = calculate_body_center(pose_data)
```

### Scoring
```python
prep_score = score_preparation_position(pose_data)
contact_score = score_contact_position(pose_data)
finish_score = score_finish_position(pose_data)
```

### Utilities
```python
angle = calculate_3d_angle(point1, point2, point3)  # Returns angle in degrees
```